#### Name: Blessing Adeniji
#### Degree: MSc Artifical Intelligence Online
#### Capstone Project: AI-Generated Text Detection - Deepfakes

##### Step 2: Finetuning small models

In [1]:
import torch

# Check if PyTorch can see the GPU
print("CUDA available:", torch.cuda.is_available())

# Print the GPU name if found
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [1]:
# Fine-tune smaller models such as Ettin-68m on ChatGPT Abstracts dataset.
# Smaller dataset means fastest run which equals to 10-20mins
import pandas as pd
from datasets import Dataset

# Load the ChatGPT Abstracts dataset
train_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_train.csv")
validation_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_val.csv")
test_dataset = pd.read_csv("data_splits/Chatgpt-Research-Abstracts_test.csv")

# Converting the pandas tables into HuggingFace dataset format
train_ds = Dataset.from_pandas(train_dataset)
validation_ds = Dataset.from_pandas(validation_dataset)
test_ds = Dataset.from_pandas(test_dataset)

# print the sizes of the datasets
print("Train dataset size:", len(train_ds))
print("Validation dataset size:", len(validation_ds))
print("Test dataset size:", len(test_ds))

Train dataset size: 14000
Validation dataset size: 3000
Test dataset size: 3000


In [2]:
# load the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# The model finetuning is ettin-encoder
model_name = "jhu-clsp/ettin-encoder-68m"

# Tokenizer converts text into tokens that the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model for sequence classification (human=0, AI=1)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-68m
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
# Tokenize the text
# Convert all texts into tokens, cutt off at 512 tokens.
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
validation_ds = validation_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [4]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin68m_abstracts",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation
     learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",            # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=10,
    report_to="none"
)   

# Pads each batch to equal length automatically, so that the model can process it. This is important for variable-length sequences.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create the Trainer
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_ds,              # training dataset
    eval_dataset=validation_ds,          # evaluation dataset
    data_collator=data_collator,         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.029950,0.041130,0.991667,0.991608
2,0.000021,0.039219,0.993000,0.992974
3,0.000010,0.042219,0.993333,0.993316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2625, training_loss=0.03193261764466955, metrics={'train_runtime': 988.864, 'train_samples_per_second': 42.473, 'train_steps_per_second': 2.655, 'total_flos': 5081803926587136.0, 'train_loss': 0.03193261764466955, 'epoch': 3.0})

In [5]:
# Evaluate the fine-tuned model on the unseen test set
results = trainer.evaluate(test_ds)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000010,0.036982,3,0.994333,0.994320


{'eval_loss': 0.03698150813579559, 'eval_accuracy': 0.9943333333333333, 'eval_f1': 0.9943200801871033}


In [6]:
# Save the fine-tuned model and tokenizer
model.save_pretrained("models/ettin68m_abstracts_final")
tokenizer.save_pretrained("models/ettin68m_abstracts_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/ettin68m_abstracts_final\\tokenizer_config.json',
 'models/ettin68m_abstracts_final\\tokenizer.json')